---
layout: post
title: Open Coding Society - Lesson Gamify
description: Start interactive experience by pressing "Play".
permalink: /Gamify-Lesson
codemirror: true
hide: true
toc: false
---

Enemy Death & Collision in MarketPirateGame
A practical guide to how enemies are detected, engaged, and destroyed in the game engine.

Overview
The game uses a proximity-based collision system rather than pixel-perfect hitbox collision. Instead of checking whether two sprites overlap on a canvas, it measures the logical distance between the player's position and each enemy's position every frame. When that distance falls below a threshold, the game treats it as a "collision" and prompts the player to fight.

1. How Enemy Position Works
Each enemy is a WorldEnemy instance. When spawned, it is given a logical X/Y position relative to the game container:
class WorldEnemy {
  constructor(enemyType, logicalX, logicalY, container) {
    this.logicalX = logicalX;
    this.logicalY = logicalY;
    this._container = container;
    // ...
  }
}These logical coordinates are then translated to screen-space pixels using the container's bounding rectangle:
_syncPosition() {
  const rect = this._container.getBoundingClientRect();
  this.el.style.left = (rect.left + this.logicalX) + 'px';
  this.el.style.top  = (rect.top  + this.logicalY) + 'px';
}Why this matters: The game world scrolls and resizes, so logical positions are relative to the container, not the viewport. syncPosition() is called every resize() to keep enemy markers correct.

2. Player Position Extraction
The player's logical position is read from the Player game object each frame inside update():
_playerLogicalPos() {
  const p = this.gameEnv.gameObjects?.find(o => o instanceof Player);
  if (!p?.position) return null;
  return {
    x: p.position.x + (p.width  || 0) * 0.5,  // horizontal center
    y: p.position.y + (p.height || 0) * 0.8,  // near the player's feet
  };
}The offset to the feet (0.8 of height) is intentional — it makes proximity feel natural because the player "approaches" an enemy from their standing position, not their center.

3. Proximity Collision Detection
Every frame, _findNearbyEnemy() loops over all living enemies and computes the Euclidean distance to the player:
_findNearbyEnemy(threshold = 120) {
  const pos = this._playerLogicalPos();
  if (!pos) return null;

  let closest = null;
  let best    = threshold;

  for (const e of this._worldEnemies) {
    if (e.defeated) continue;                            // skip dead enemies

    const d = Math.hypot(pos.x - e.logicalX, pos.y - e.logicalY);

    if (d < best) {
      best    = d;
      closest = e;
    }
  }
  return closest; // null if no enemy is within 120px
}   Value Meaning     threshold = 120 Detection radius in logical pixels   Math.hypot(dx, dy) True 2D distance (Pythagoras theorem)   Returns closest Only the single nearest enemy matters   If the returned value is not null, the HUD displays a prompt and pressing E triggers the battle.

4. The update() Loop
update() runs every game tick and ties everything together:
update() {
  if (!this._gameStarted) return;

  const player = this.gameEnv.gameObjects.find(o => o instanceof Player);
  if (!player) return;

  // Don't check collisions while a shop or battle is open
  if (this._open || this._battleOpen || this._specOpen) {
    this._setHint('');
    this._worldEnemies.forEach(e => e.hideHint());
    return;
  }

  const nearby = this._findNearbyEnemy(120);
  this._nearbyEnemy = nearby;

  this._worldEnemies.forEach(e => e.hideHint());

  if (nearby) {
    nearby.showHint('Press E to fight');
    this._setHint(`⚔ ${nearby.type.name} — Lv.${nearby.type.level} · Press E to fight`);
  }
  // ...
}Key guards:

If _gameStarted is false (still on the menu), nothing runs.
If a UI panel is open, collision is suspended so the player can't accidentally trigger a second battle.


5. Triggering the Battle
When the player presses E, the key handler checks _nearbyEnemy:
this._keyHandler = (e) => {
  if (e.key !== 'e' && e.key !== 'E') return;
  if (this._battleOpen || this._open || this._specOpen) return;

  if (this._nearbyEnemy) {
    this._startBattle(this._nearbyEnemy);  // collision → fight!
    return;
  }
  // otherwise check shop zones...
};_startBattle() opens the BattleUI modal and passes through the player's current stats:
_startBattle(worldEnemy) {
  if (this._battleOpen) return;
  this._battleOpen = true;
  this._nearbyEnemy = null;

  this._battleUI = new BattleUI(
    worldEnemy.type,
    (rubyReward, wasDefeated, remainingHp, xpGained) => {
      // --- battle callback ---
      this._battleOpen = false;
      this._battleUI   = null;

      if (wasDefeated) {
        this._showGameOver();
      } else {
        this._playerCurHp = Math.max(1, remainingHp);
        this._updateWorldHP();

        if (rubyReward > 0) {
          worldEnemy.markDefeated();  // ← enemy death happens here
          this._killCount++;
          this._bankedRubies += rubyReward;
        }
        if (xpGained > 0) this._gainXp(xpGained);
      }
    },
    this._playerCurHp,
    this._playerLevel,
    this._playerAtk,
    this._playerDef,
    this._playerMaxHp,
  );
}

6. Enemy Death — markDefeated()
When the player wins a battle and earns rubies, worldEnemy.markDefeated() is called on the WorldEnemy instance:
markDefeated() {
  this.defeated = true;   // flag prevents further collision checks
  this.hideHint();        // remove the tooltip bubble
  this.el.classList.add('defeated');  // triggers CSS fade-out

  setTimeout(() => this.el.remove(), 400);  // remove from DOM after animation
}The CSS that drives the visual death:
.world-enemy.defeated {
  opacity: 0;
  pointer-events: none;  /* no further mouse interaction */
}The transition: opacity .3s on .world-enemy makes this a smooth fade rather than an instant disappearance.
The defeated flag is what stops the enemy from being detected again:
for (const e of this._worldEnemies) {
  if (e.defeated) continue;  // ← dead enemies are skipped
  // ...
}The enemy object remains in _worldEnemies array briefly after death, but the flag prevents any interaction. The respawn timer cleans them out:
this._respawnTimer = setInterval(() => {
  this._worldEnemies = this._worldEnemies.filter(e => !e.defeated);
  if (this._worldEnemies.length < 4) this._spawnEnemies(3);
}, 25000);

7. Spawning & Respawning
Enemies are spawned with a guard that keeps them away from all market zones so they don't overlap with shops:
_spawnEnemies(n) {
  const allZones = [
    { cx: this.shopZone.x + this.shopZone.width  / 2,
      cy: this.shopZone.y + this.shopZone.height / 2 },
    ...this._specZones.map(z => ({ cx: z.cx, cy: z.cy })),
  ];

  for (let i = 0; i < n; i++) {
    let lx, ly, tries = 0;
    do {
      lx = margin + Math.random() * (width  - margin * 2);
      ly = margin + Math.random() * (height - margin * 2);
      tries++;
    } while (tries < 30 && allZones.some(z => Math.hypot(lx - z.cx, ly - z.cy) < 120));

    this._worldEnemies.push(new WorldEnemy(type, lx, ly, container));
  }
}The do...while retries up to 30 times to find a position that is at least 120px away from every market zone center.

8. Full Death Lifecycle
Player walks near enemy
        │
        ▼
_findNearbyEnemy() returns WorldEnemy   (distance < 120px)
        │
        ▼
Player presses E
        │
        ▼
_startBattle(worldEnemy)  →  BattleUI opens
        │
   [player wins]
        │
        ▼
Battle callback fires
        ├── worldEnemy.markDefeated()
        │         ├── this.defeated = true
        │         ├── CSS fade-out (.defeated class)
        │         └── DOM removal after 400ms
        │
        ├── _killCount++
        ├── rubies banked
        └── _gainXp(xpGained)

After 25 seconds:
_worldEnemies filtered (removes defeated entries)
If count < 4 → _spawnEnemies(3)

9. Quick Reference
   Concept Where Key Detail     Enemy position WorldEnemy._syncPosition() Logical coords + container rect offset   Player position _playerLogicalPos() Feet position (0.8 height)   Collision check _findNearbyEnemy(120) Euclidean distance, runs every frame   Battle trigger _keyHandler → _startBattle() Only on E keypress within range   Enemy death flag WorldEnemy.defeated Skips enemy in all future checks   Visual death .defeated CSS class opacity: 0 + pointer-events: none   DOM cleanup setTimeout(() => el.remove(), 400) After fade animation completes   Array cleanup _respawnTimer filter Every 25 seconds   Spawn guard do...while with zone check Keeps enemies 120px from all shops

In [ ]:
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>OCS Engine | Professional Edition</title>
    <link rel="stylesheet" href="style.css">
</head>
<body>
    <div id="project-launcher">
        <div class="launcher-sidebar">
            <div class="logo">OCS<span>Hub</span></div>
            <button onclick="showHubSection('projects')" class="nav-btn active">📁 Projects</button>
            <button onclick="showHubSection('docs')" class="nav-btn">📖 Documentation</button>
        </div>
        <div class="launcher-content">
            <div id="hub-projects">
                <div class="launcher-header">
                    <h2>Your Projects</h2>
                    <button onclick="createNewProject()" class="btn-run">+ New Project</button>
                </div>
                <div id="project-list"></div>
            </div>

            <div id="hub-docs" class="hidden">
                <div class="launcher-header">
                    <h2>Engine Documentation</h2>
                </div>
                <div id="docs-render-area" class="docs-container"></div>
            </div>
        </div>
    </div>

    <div class="top-bar">
        <div class="logo" onclick="location.reload()" style="cursor:pointer">OCS<span>Engine</span></div>
        <div class="controls">
            <button onclick="saveProject()" class="btn-save">💾 Save</button>
            <button onclick="startTutorial()" class="btn-tut">🎓 Tutorial</button>
            <button onclick="runGame()" class="btn-run">▶ Play Scene</button>
        </div>
    </div>

    <div class="editor-layout">
        <div class="dock">
            <div class="dock-header">FileSystem</div>
            <div class="asset-controls">
                <button class="btn-tiny" onclick="createNewFolder()">+ Folder</button>
                <button class="btn-tiny" onclick="document.getElementById('spriteInput').click()">+ Sprite</button>
            </div>
            <input type="file" id="spriteInput" style="display:none" accept="image/*">
            <div id="file-tree"></div>
        </div>

        <div class="center-view">
            <div class="viewport">
                <canvas id="gameCanvas" tabindex="1"></canvas>
            </div>
            <div class="code-area">
                <div class="tabs"><span id="active-project-name">main.js</span> <span id="save-status"></span></div>
                <textarea id="codeEditor" spellcheck="false" placeholder="// Click 'Tutorial' or select a project!"></textarea>
            </div>
        </div>

        <div class="dock">
            <div class="dock-header">Scene Hierarchy</div>
            <div id="hierarchy-list"></div>

            <div class="dock-header">Inspector</div>
            <div id="inspector-content">
                <div class="inspector-section">
                    <div class="prop-row"><span class="prop-label">State</span><span class="prop-value" id="live-status">Stopped</span></div>
                    <div class="prop-row"><span class="prop-label">Objects</span><span class="prop-value" id="live-count">0</span></div>
                </div>
                <div class="inspector-section">
                    <p class="section-title">LIVE TRANSFORM</p>
                    <div class="prop-row"><span class="prop-label">X Pos</span><span class="prop-value" id="live-x">0</span></div>
                    <div class="prop-row"><span class="prop-label">Y Pos</span><span class="prop-value" id="live-y">0</span></div>
                </div>
            </div>
        </div>
    </div>

    <div id="tutorial-box" class="hidden">
        <div class="tut-header">
            <h3 id="tut-title">Step 1</h3>
            <button onclick="document.getElementById('tutorial-box').classList.add('hidden')">×</button>
        </div>
        <div id="tut-text"></div>
        <button onclick="nextStep()" class="btn-small">Next Lesson →</button>
    </div>

    <script src="engine.js"></script>
    <script src="docs.js"></script>
</body>
</html>